# 02 · Train the LSTM model (GPU)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MR-SAIRAM-7/tokenEdge/blob/main/notebooks/02_train_lstm.ipynb)

A 2-layer LSTM reads the last *N* bars of scaled features and predicts P(price up). It is trained in PyTorch here and exported to **ONNX**, so the web app runs it with ONNX Runtime and never needs PyTorch.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Leakage controls: data after the validation window is never loaded; the feature scaler is fitted on training rows only; early stopping and the trading threshold use validation data only; the ONNX export is checked against PyTorch.

In [ ]:
# --- Setup: works in Google Colab and locally (run from the repo or its notebooks/ folder)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/tokenEdge"):
        !git clone -q https://github.com/MR-SAIRAM-7/tokenEdge.git /content/tokenEdge
    %cd /content/tokenEdge
    !pip install -q -e ".[train]"
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..
import tokenedge
print("TokenEdge", tokenedge.__version__)

> If pip upgrades numpy or pandas, Colab may ask to restart. Use **Runtime → Restart session** and run the cells again (the clone and install are skipped the second time).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (slow but works)")

## Configuration

In [ ]:
SYMBOL = "SPY"                 # e.g. "BTC/USDT" with SOURCE="ccxt", INTERVAL="1h"
SOURCE = "yfinance"
INTERVAL = "1d"
LABEL_METHOD = "fixed_horizon" # or "triple_barrier"
SEQ_LEN = None                 # None = default (60 for 1d, 48 for 1h)
HIDDEN, LAYERS, DROPOUT = 64, 2, 0.2
LR, BATCH_SIZE, MAX_EPOCHS, PATIENCE = 1e-3, 256, 100, 10
REFRESH_DATA = False

In [ ]:
from tokenedge.config import default_splits, default_label_config
from tokenedge.data import get_ohlcv

df = get_ohlcv(SYMBOL, SOURCE, INTERVAL, refresh=REFRESH_DATA)
print(f"{len(df):,} bars  {df.index[0]} -> {df.index[-1]}  | splits: {default_splits(SOURCE)}")

## Train

In [ ]:
from tokenedge.ml.lstm_model import train_lstm

out = train_lstm(
    df, SYMBOL, SOURCE, INTERVAL,
    label_config=default_label_config(LABEL_METHOD, INTERVAL),
    seq_len=SEQ_LEN, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT,
    lr=LR, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, patience=PATIENCE,
)
md = out.metadata
print("model_id:", md["model_id"], "| ONNX vs PyTorch max diff:", md["onnx_parity_max_abs_diff"])

## Training curve and validation results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

h = md["history"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(h["train_loss"], label="train", color="#2a78d6")
axes[0].plot(h["val_loss"], label="validation", color="#1baf7a")
axes[0].set_title("Loss per epoch"); axes[0].set_xlabel("epoch"); axes[0].legend()
t = pd.DataFrame(md["threshold_search"])
axes[1].plot(t["threshold"], t["sharpe"], marker="o", color="#1baf7a")
axes[1].axvline(md["threshold"]["long"], color="gray", lw=1)
axes[1].set_title("Validation Sharpe after costs vs threshold")
plt.tight_layout(); plt.show()

val = md["metrics"]["val"]
pd.Series({k: val[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "log_loss", "base_rate", "sharpe_after_costs"]}).round(4)

## Export

In [ ]:
from tokenedge.ml.training import zip_model_dir

archive = zip_model_dir(out.model_dir)
print("created", archive, "| files:", sorted(p.name for p in out.model_dir.iterdir()))
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))

## Next steps

Unzip into the project's `models/` folder (`model.onnx`, `scaler.json`, `metadata.json`), refresh the web app, then compare it against SMA and XGBoost on the **Compare** page. Commit the folder to include it in the deployed demo.